In [0]:
%sql
CREATE OR REPLACE TABLE medical_insurance.gold.drug_utilization AS
WITH drug_base AS (
  SELECT 
    d.drug_id,
    d.drug_name,
    d.generic_name,
    d.manufacturer,
    d.drug_category,
    d.unit_amount
  FROM medical_insurance.silver.drug_silver d
),
prescription_stats AS (
  SELECT 
    pi.drug_id,
    COUNT(DISTINCT pi.prescription_item_id) AS total_prescriptions,
    COUNT(DISTINCT pi.prescription_id) AS unique_prescriptions,
    SUM(pi.quantity) AS total_quantity_prescribed,
    AVG(pi.quantity) AS avg_quantity_per_prescription,
    AVG(pi.duration_days) AS avg_duration_days,
    COUNT(DISTINCT CASE WHEN pi.frequency = 'Once Daily' THEN pi.prescription_item_id END) AS once_daily_count,
    COUNT(DISTINCT CASE WHEN pi.frequency = 'Twice Daily' THEN pi.prescription_item_id END) AS twice_daily_count,
    COUNT(DISTINCT CASE WHEN pi.frequency = 'Three Times Daily' THEN pi.prescription_item_id END) AS thrice_daily_count
  FROM medical_insurance.silver.prescription_items_silver pi
  GROUP BY pi.drug_id
),
inventory_stats AS (
  SELECT 
    i.drug_id,
    COUNT(DISTINCT i.hospital_id) AS hospitals_stocking,
    SUM(i.quantity_available) AS total_quantity_available,
    AVG(i.quantity_available) AS avg_quantity_per_hospital,
    SUM(i.reorder_level) AS total_reorder_level,
    AVG(i.reorder_level) AS avg_reorder_level,
    MIN(i.expiration_date) AS earliest_expiration_date,
    MAX(i.expiration_date) AS latest_expiration_date,
    COUNT(DISTINCT CASE WHEN i.quantity_available < i.reorder_level THEN i.inventory_id END) AS low_stock_count
  FROM medical_insurance.silver.drug_inventory_silver i
  GROUP BY i.drug_id
),
transaction_stats AS (
  SELECT 
    dt.drug_id,
    COUNT(DISTINCT dt.transaction_id) AS total_transactions,
    COUNT(DISTINCT CASE WHEN dt.transaction_type = 'Purchase' THEN dt.transaction_id END) AS purchase_transactions,
    COUNT(DISTINCT CASE WHEN dt.transaction_type = 'Dispensing' THEN dt.transaction_id END) AS dispensing_transactions,
    SUM(CASE WHEN dt.transaction_type = 'Purchase' THEN dt.quantity ELSE 0 END) AS total_purchased_quantity,
    SUM(CASE WHEN dt.transaction_type = 'Dispensing' THEN dt.quantity ELSE 0 END) AS total_dispensed_quantity,
    MIN(dt.transaction_date) AS first_transaction_date,
    MAX(dt.transaction_date) AS last_transaction_date
  FROM medical_insurance.silver.drug_transaction_silver dt
  GROUP BY dt.drug_id
)
SELECT 
  db.drug_id,
  db.drug_name,
  db.generic_name,
  db.manufacturer,
  db.drug_category,
  ROUND(db.unit_amount, 2) AS unit_price,
  
  -- Prescription metrics
  COALESCE(ps.total_prescriptions, 0) AS total_prescriptions,
  COALESCE(ps.unique_prescriptions, 0) AS unique_prescriptions,
  COALESCE(ps.total_quantity_prescribed, 0) AS total_quantity_prescribed,
  ROUND(COALESCE(ps.avg_quantity_per_prescription, 0), 2) AS avg_quantity_per_prescription,
  ROUND(COALESCE(ps.avg_duration_days, 0), 2) AS avg_treatment_duration_days,
  COALESCE(ps.once_daily_count, 0) AS once_daily_prescriptions,
  COALESCE(ps.twice_daily_count, 0) AS twice_daily_prescriptions,
  COALESCE(ps.thrice_daily_count, 0) AS thrice_daily_prescriptions,
  
  -- Inventory metrics
  COALESCE(inv.hospitals_stocking, 0) AS hospitals_stocking_drug,
  COALESCE(inv.total_quantity_available, 0) AS current_stock_quantity,
  ROUND(COALESCE(inv.avg_quantity_per_hospital, 0), 2) AS avg_stock_per_hospital,
  COALESCE(inv.total_reorder_level, 0) AS total_reorder_level,
  ROUND(COALESCE(inv.avg_reorder_level, 0), 2) AS avg_reorder_level,
  inv.earliest_expiration_date,
  inv.latest_expiration_date,
  COALESCE(inv.low_stock_count, 0) AS low_stock_locations,
  
  -- Transaction metrics
  COALESCE(ts.total_transactions, 0) AS total_transactions,
  COALESCE(ts.purchase_transactions, 0) AS purchase_transactions,
  COALESCE(ts.dispensing_transactions, 0) AS dispensing_transactions,
  COALESCE(ts.total_purchased_quantity, 0) AS total_purchased_quantity,
  COALESCE(ts.total_dispensed_quantity, 0) AS total_dispensed_quantity,
  ts.first_transaction_date,
  ts.last_transaction_date,
  
  -- Financial metrics
  ROUND(COALESCE(ps.total_quantity_prescribed * db.unit_amount, 0), 2) AS total_prescription_value,
  ROUND(COALESCE(inv.total_quantity_available * db.unit_amount, 0), 2) AS current_inventory_value,
  
  -- Utilization metrics
  CASE 
    WHEN ts.total_dispensed_quantity > 0 AND inv.total_quantity_available > 0 
    THEN ROUND((ts.total_dispensed_quantity * 100.0 / (ts.total_dispensed_quantity + inv.total_quantity_available)), 2)
    ELSE 0
  END AS utilization_rate_pct,
  
  CASE 
    WHEN ts.total_purchased_quantity > 0 AND ts.total_dispensed_quantity > 0
    THEN ROUND((ts.total_dispensed_quantity * 100.0 / ts.total_purchased_quantity), 2)
    ELSE 0
  END AS turnover_rate_pct,
  
  -- Categorizations
  CASE 
    WHEN ps.total_prescriptions > 1000 THEN 'High Demand'
    WHEN ps.total_prescriptions BETWEEN 500 AND 1000 THEN 'Medium Demand'
    WHEN ps.total_prescriptions < 500 THEN 'Low Demand'
    ELSE 'No Demand'
  END AS demand_category,
  
  CASE 
    WHEN inv.low_stock_count > 0 THEN 'Stock Alert'
    WHEN inv.total_quantity_available < inv.total_reorder_level THEN 'Reorder Needed'
    WHEN inv.total_quantity_available >= inv.total_reorder_level THEN 'Adequate Stock'
    ELSE 'Unknown'
  END AS stock_status,
  
  CASE 
    WHEN DATEDIFF(inv.earliest_expiration_date, CURRENT_DATE()) < 30 THEN 'Expiring Soon'
    WHEN DATEDIFF(inv.earliest_expiration_date, CURRENT_DATE()) BETWEEN 30 AND 90 THEN 'Monitor Expiration'
    WHEN DATEDIFF(inv.earliest_expiration_date, CURRENT_DATE()) > 90 THEN 'Good Shelf Life'
    ELSE 'Unknown'
  END AS expiration_status,
  
  CURRENT_TIMESTAMP() AS created_at
  
FROM drug_base db
LEFT JOIN prescription_stats ps ON db.drug_id = ps.drug_id
LEFT JOIN inventory_stats inv ON db.drug_id = inv.drug_id
LEFT JOIN transaction_stats ts ON db.drug_id = ts.drug_id

In [0]:
%sql
-- Display sample records from drug utilization gold table
SELECT 
  drug_name,
  drug_category,
  total_prescriptions,
  current_stock_quantity,
  total_prescription_value,
  demand_category,
  stock_status,
  expiration_status
FROM medical_insurance.gold.drug_utilization
ORDER BY total_prescriptions DESC
LIMIT 10

In [0]:
%sql
-- Summary statistics by drug category and demand
SELECT 
  drug_category,
  demand_category,
  COUNT(DISTINCT drug_id) AS total_drugs,
  ROUND(AVG(total_prescriptions), 2) AS avg_prescriptions,
  ROUND(AVG(current_stock_quantity), 2) AS avg_stock,
  ROUND(SUM(total_prescription_value), 2) AS total_prescription_value,
  ROUND(SUM(current_inventory_value), 2) AS total_inventory_value,
  ROUND(AVG(turnover_rate_pct), 2) AS avg_turnover_rate,
  COUNT(DISTINCT CASE WHEN stock_status = 'Stock Alert' THEN drug_id END) AS drugs_with_stock_alert
FROM medical_insurance.gold.drug_utilization
GROUP BY drug_category, demand_category
ORDER BY drug_category, demand_category